# Visual-language assistant with Gemma 4 and OpenVINO
![](https://ai.google.dev/gemma/images/gemma4_banner.png)

[Gemma 4](https://huggingface.co/collections/google/gemma-4) is Google DeepMind's latest generation of open multimodal models.
It handles **text and image** input (with **audio** on smaller models) and generates text output.

The family includes four sizes:

| Model | Total Params | Effective / Active | Modalities | Context | Architecture |
|---|---|---|---|---|---|
| **E2B** | 5.1 B | 2.3 B effective | Text, Image, Audio | 128 K | Dense + PLE |
| **E4B** | 8 B | 4.5 B effective | Text, Image, Audio | 128 K | Dense + PLE |
| **26B‑A4B** | 25.2 B | 3.8 B active | Text, Image | 256 K | MoE (8 / 128 experts) |
| **31B** | 30.7 B | 30.7 B | Text, Image | 256 K | Dense |

Key enhancements over Gemma 3:
- **Thinking mode** – built-in chain-of-thought reasoning via `enable_thinking=True`
- **Native system prompt** – first-class `system` role support
- **Interleaved multimodal input** – freely mix multiple images and text in any order
- **MoE architecture** – 26B‑A4B runs nearly as fast as a 4B model
- **Extended context** – up to 256 K tokens on larger models

In this notebook we will:
1. Export & compress a Gemma 4 model to OpenVINO IR with [Optimum Intel](https://github.com/huggingface/optimum-intel)
2. Run image-understanding inference via `OVModelForVisualCausalLM`
3. Demonstrate **native system prompt** support — control model persona via `system` role
4. Demonstrate **interleaved multi-image** input — compare and analyze multiple images
5. Demonstrate **thinking mode** (chain-of-thought reasoning)
6. Launch an interactive Gradio chat demo


#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Select model](#Select-model)
- [Convert and Optimize model](#Convert-and-Optimize-model)
  - [Select inference device](#Select-inference-device)
- [Run model inference](#Run-model-inference)
  - [Image understanding](#Image-understanding)
  - [Native system prompt](#Native-system-prompt)
  - [Interleaved multimodal input](#Interleaved-multimodal-input)
  - [Thinking mode](#Thinking-mode)
- [Interactive demo](#Interactive-demo)


⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO. It may be fully supported and validated in the future.

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).


<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/gemma4/gemma4.ipynb" />

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [ ]:
%pip uninstall -q -y optimum
%pip install -q "git+https://github.com/rkazants/optimum-intel.git@support_gemma4" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -qU --pre "openvino" "nncf" --extra-index-url https://storage.openvinotoolkit.org/simple/wheels/nightly
%pip install -q "transformers==5.5.0"
%pip install -q "torch>=2.10" "torchvision" "Pillow" "gradio>=6.0" "opencv-python" "requests" --extra-index-url https://download.pytorch.org/whl/cpu

In [ ]:
from pathlib import Path
import requests

if not Path("cmd_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py")
    open("cmd_helper.py", "w").write(r.text)

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w").write(r.text)

## Select model
[back to top ⬆️](#Table-of-contents:)

Several Gemma 4 instruction-tuned models are available in the [Gemma 4 collection](https://huggingface.co/collections/google/gemma-4).

| Model | Notes |
|---|---|
| `google/gemma-4-E2B-it` | 5.1 B total / 2.3 B effective, supports audio, best for quick testing |
| `google/gemma-4-E4B-it` | 8 B total / 4.5 B effective, supports audio |
| `google/gemma-4-26B-A4B-it` | 25.2 B total / 3.8 B active (MoE), fast inference |
| `google/gemma-4-31B-it` | 30.7 B dense, highest quality |

> **Note:** Larger models require significantly more RAM.

In [ ]:
import ipywidgets as widgets

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("gemma4.ipynb")

model_ids = [
    "google/gemma-4-E2B-it",
    "google/gemma-4-E4B-it",
    "google/gemma-4-26B-A4B-it",
    "google/gemma-4-31B-it",
]

model_id = widgets.Dropdown(
    options=model_ids,
    default=model_ids[0],
    description="Model:",
)

model_id

In [ ]:
print(f"Selected {model_id.value}")
pt_model_id = model_id.value
model_dir = Path(pt_model_id.split("/")[-1])

## Convert and Optimize model
[back to top ⬆️](#Table-of-contents:)

Gemma 4 is a PyTorch model. We convert it to OpenVINO Intermediate Representation (IR) using
[🤗 Optimum Intel](https://huggingface.co/docs/optimum/intel/index) which provides a simple CLI:

```bash
optimum-cli export openvino --model <model_id_or_path> --task <task> --weight-format <format> <output_dir>
```

Weight compression via [NNCF](https://github.com/openvinotoolkit/nncf) reduces model size with minimal quality loss, making it practical to run large models on consumer hardware. Available formats: **FP16** (no compression), **INT8** (~2× smaller), **INT4** (~4× smaller).

In [ ]:
weight_format = widgets.Dropdown(
    options=["FP16", "INT8", "INT4"],
    value="INT8",
    description="Weight format:",
)

weight_format

In [ ]:
from cmd_helper import optimum_cli

model_export_dir = model_dir / weight_format.value

if not model_export_dir.exists():
    optimum_cli(pt_model_id, model_export_dir, additional_args={"task": "image-text-to-text", "weight-format": weight_format.value.lower()})

### Select inference device
[back to top ⬆️](#Table-of-contents:)

We use `OVModelForVisualCausalLM` from [Optimum Intel](https://huggingface.co/docs/optimum/intel/index)
for inference. It loads the exported OpenVINO IR model and provides a familiar
HuggingFace `generate()` API with automatic tokenization via `AutoProcessor`.

In [ ]:
from notebook_utils import device_widget

device = device_widget(default="AUTO", exclude=["NPU", "GPU"])

device

## Run model inference
[back to top ⬆️](#Table-of-contents:)

In [ ]:
from optimum.intel.openvino import OVModelForVisualCausalLM
from transformers import AutoProcessor
from PIL import Image

In [ ]:
def load_image(path_or_url):
    import os
    from io import BytesIO

    if isinstance(path_or_url, str) and path_or_url.startswith(("http", "https")):
        import requests

        response = requests.get(path_or_url)
        if response.status_code != 200 or not response.content:
            raise FileNotFoundError(f"Could not retrieve image from URL: {path_or_url}")
        try:
            return Image.open(BytesIO(response.content)).convert("RGB")
        except Exception as e:
            raise FileNotFoundError(f"Failed to open image from URL: {path_or_url}") from e
    else:
        file_path = str(path_or_url)
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Image file does not exist: {file_path}")
        try:
            return Image.open(file_path).convert("RGB")
        except Exception as e:
            raise FileNotFoundError(f"Failed to open image file: {file_path}") from e


image_url = "https://github.com/openvinotoolkit/openvino_notebooks/assets/29454499/d5fbbd1a-d484-415c-88cb-9986625b7b11"
image_file = Path("cat.png")

if not image_file.exists():
    image = load_image(image_url)
    image.save(image_file)
else:
    image = load_image(image_file)

In [ ]:
processor = AutoProcessor.from_pretrained(model_export_dir)
model = OVModelForVisualCausalLM.from_pretrained(model_export_dir, device=device.value)

### Image understanding

Gemma 4 can analyze images at variable resolutions. Place image content **before** text
in the prompt for best results.

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "Describe what is unusual in these images."},
        ],
    }
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
inputs = processor(text=text, images=[image], return_tensors="pt")
input_len = inputs["input_ids"].shape[-1]

display(image)
output = model.generate(**inputs, do_sample=False, max_new_tokens=100)
response = processor.decode(output[0][input_len:], skip_special_tokens=True)
print(response)

### Native system prompt

Unlike Gemma 3, Gemma 4 introduces first-class support for the **`system` role**.
This allows you to control the model's persona and behavior by adding a `system`
message at the start of the conversation — no need to embed instructions in the user message.

Let's try it — we'll give the model a professional art-critic persona and see how it changes the response.

In [ ]:
# Gemma 4 natively supports the system role
messages = [
    {"role": "system", "content": "You are a professional art critic. Analyze visual compositions with attention to color, form, and emotional impact."},
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "What do you see in this image?"},
        ],
    },
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
inputs = processor(text=text, images=[image], return_tensors="pt")
input_len = inputs["input_ids"].shape[-1]

output = model.generate(**inputs, do_sample=False, max_new_tokens=200)
response = processor.decode(output[0][input_len:], skip_special_tokens=True)
print(response)

### Interleaved multimodal input

Gemma 4 can process **multiple images freely mixed with text** in a single prompt.
This enables tasks like visual comparison, multi-page document analysis, and combining information from several sources.

Let's download a second image and ask the model to compare both.

In [ ]:
image2_url = "https://github.com/user-attachments/assets/da3edb79-ae36-4973-9eaf-6ef712425faa"
image2_file = Path("sunset.png")

if not image2_file.exists():
    image2 = load_image(image2_url)
    image2.save(image2_file)
else:
    image2 = load_image(image2_file)

# Pass multiple images — Gemma 4 handles interleaved multimodal input
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "image", "image": image2},
            {"type": "text", "text": "I have two images. Compare them: describe what you see in each and which image has a warmer color palette."},
        ],
    }
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
inputs = processor(text=text, images=[image, image2], return_tensors="pt")
input_len = inputs["input_ids"].shape[-1]

display(image, image2)
output = model.generate(**inputs, do_sample=False, max_new_tokens=200)
response = processor.decode(output[0][input_len:], skip_special_tokens=True)
print(response)

### Thinking mode

Gemma 4 supports built-in **chain-of-thought reasoning**. When `enable_thinking=True`
is passed to `apply_chat_template`, the model first reasons internally
and then produces a final answer.

The output structure looks like:
```
<|channel>thought
[internal reasoning]
<channel|>
[final answer]
```

`processor.parse_response()` conveniently separates the thinking block from the answer.

In [ ]:
# Enable thinking mode via enable_thinking=True in chat template
messages = [
    {"role": "system", "content": "You are a helpful assistant. Think step by step before answering."},
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "Look at this image. Count all the visible animals and describe each one."},
        ],
    },
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)
inputs = processor(text=text, images=[image], return_tensors="pt")
input_len = inputs["input_ids"].shape[-1]

output = model.generate(**inputs, do_sample=False, max_new_tokens=500)
response = processor.decode(output[0][input_len:], skip_special_tokens=False)

print("=== Raw response ===")
print(response)
print("\n=== Parsed response ===")
parsed = processor.parse_response(response)
print(parsed)

## Interactive demo
[back to top ⬆️](#Table-of-contents:)

In [ ]:
if not Path("gradio_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/gemma4/gradio_helper.py")
    open("gradio_helper.py", "w").write(r.text)

In [ ]:
from gradio_helper import make_demo

demo = make_demo(model, processor)

try:
    demo.launch(debug=True, height=900)
except Exception:
    demo.launch(share=True, debug=True, height=900)
# if you are launching remotely, specify server_name and server_port
# demo.launch(server_name='your server name', server_port='server port in int')
# Read more in the docs: https://gradio.app/docs/